In [57]:
import math
import numpy as np
import pandas as pd
import scipy.stats as stats
from pandas import read_csv
from scipy.optimize import minimize
from scipy.stats import spearmanr

def CoSMoS2s(data):
    
    # ------------------------------------------------------------------
    # Helper functions
    # ------------------------------------------------------------------
    def spearman_acf(series, max_lag=None):
        series = np.asarray(series)

        if max_lag is None:
            max_lag = int(np.floor(len(series) / 4))

        acf_values = np.empty(max_lag)

        for lag in range(max_lag):
            if lag == 0:
                corr = spearmanr(series, series)[0]
                acf_values[lag] = np.nan_to_num(corr, nan=1.0)
            else:
                x = series[:-lag]
                y = series[lag:]
                corr = spearmanr(x, y)[0]
                acf_values[lag] = np.nan_to_num(corr, nan=0.0)

        return acf_values

    def spearman_cross_positive_acf(series, max_lag=None):
        series = np.asarray(series)

        if max_lag is None:
            max_lag = int(np.floor(len(series) / 4))

        acf_values = np.empty(max_lag)

        for lag in range(max_lag):
            if lag == 0:
                corr = spearmanr(series, series)[0]
                acf_values[lag] = np.nan_to_num(corr, nan=1.0)
            else:
                x = series[:-lag]
                y = series[lag:]
                mask = (x > 0) & (y > 0)

                if mask.sum() < 2:
                    acf_values[lag] = 0.0
                else:
                    corr = spearmanr(x[mask], y[mask])[0]
                    acf_values[lag] = np.nan_to_num(corr, nan=0.0)

        return acf_values

    def actf_binary(rB, p0):
        c = 0.81 * (((p0 * (1 - p0)) ** 0.5) + ((p0 * (1 - p0)) ** 0.1))
        out = 1 - (1 - rB ** c) ** 2
        return np.round(out, 2)

    def ecdf(x):
        sorted_x = np.sort(x)
        probs = stats.rankdata(sorted_x, method="min") / (len(x) + 1)
        return pd.DataFrame({"p": probs, "value": sorted_x})

    def mae(x, y):
        return np.sum(np.abs(x - y)) / len(y)

    # ------------------------------------------------------------------
    # Input checks
    # ------------------------------------------------------------------
    if not isinstance(data, pd.DataFrame):
        raise TypeError("data deve essere un pandas.DataFrame con due colonne: Time e Value.")

    if data.shape[1] < 2:
        raise ValueError("data deve avere almeno due colonne: Time e Value.")

    lags=4
    p=3
    n_years=30
    # Copia e standardizzazione colonne
    data = data.iloc[:, :2].copy()
    data.columns = ["Time", "Value"]
    data["Time"] = pd.to_datetime(data["Time"])
    data["Value"] = pd.to_numeric(data["Value"], errors="coerce")
    data["Value"] = data["Value"].fillna(data["Value"].mean())
    data["month"] = data["Time"].dt.month

    # ------------------------------------------------------------------
    # STEP 1 - analyzeTS
    # ------------------------------------------------------------------
    stratified_data = {}
    p0 = {}
    pars_non_zero_values = {}
    u_t = {}
    theoretic = {}
    no0values = {}
    corTs = {}
    corCTs = {}
    corBTs = {}
  
    # DJF, MAM, JJA, SON
    month_groups = [(1, 2, 12), (3, 4, 5), (6, 7, 8), (9, 10, 11)]

    for season_idx, months_in_season in enumerate(month_groups, start=1):
        data_by_season = data[data["month"].isin(months_in_season)].copy()
        non_zero_values = data_by_season[data_by_season["Value"] != 0].copy()
        val = non_zero_values["Value"].to_numpy()

        if len(val) == 0:
            raise ValueError(f"Nessun valore non nullo trovato per la stagione {season_idx}.")

        def objective_fit_dist(par):
            a, loc, scale = par
            edf = ecdf(val)
            empirical_p = edf["p"].to_numpy()
            fitted_p = stats.gamma.cdf(edf["value"], a=a, loc=loc, scale=scale)
            return mae(empirical_p, fitted_p)

        guess_g = [1.0, 0.0, 1.0]
        bounds = ((1e-6, 10.0), (0.0, 0.0), (1e-6, 50.0))

        res = minimize(
            objective_fit_dist,
            guess_g,
            bounds=bounds,
            method="nelder-mead"
        )

        pars = np.round(res.x, 2)

        theoretical = stats.gamma.pdf(
            non_zero_values["Value"],
            a=pars[0],
            loc=pars[1],
            scale=pars[2]
        )

        stratified_data[season_idx] = data_by_season
        theoretic[season_idx] = theoretical
        no0values[season_idx] = non_zero_values
        pars_non_zero_values[season_idx] = pars

        zero_prob = (data_by_season["Value"] == 0).sum() / len(data_by_season)
        p0[season_idx] = np.round(zero_prob, 3)

        bts = data_by_season["Value"].to_numpy(copy=True)
        bts[bts > 0] = 1
        u_t[season_idx] = bts

        corTs[season_idx] = spearman_acf(data_by_season["Value"], lags)
        corBTs[season_idx] = spearman_acf(bts, lags)
        corCTs[season_idx] = spearman_cross_positive_acf(data_by_season["Value"], lags)

    analysis_result = {
        "stratified_data": stratified_data,
        "p0": p0,
        "pars_non_zero_values": pars_non_zero_values,
        "u_t": u_t,
        "theoretic": theoretic,
        "no0values": no0values,
        "corTs": corTs,
        "corCTs": corCTs,
        "corBTs": corBTs,
    }

    # ------------------------------------------------------------------
    # STEP 2 - SimulatedTS
    # ------------------------------------------------------------------
    p_0 = analysis_result["p0"]
    pars = analysis_result["pars_non_zero_values"]
    cor_CTS = analysis_result["corCTs"]
    cor_BTS = analysis_result["corBTs"]

    # mappatura stagione -> mese
    pars_by_month = {}
    p0_by_month = {}
    corCTS_by_month = {}
    corBTS_by_month = {}

    for i, months_in_season in enumerate(month_groups, start=1):
        for month in months_in_season:
            pars_by_month[month] = pars[i]
            p0_by_month[month] = p_0[i]
            corCTS_by_month[month] = cor_CTS[i]
            corBTS_by_month[month] = cor_BTS[i]

    pars_by_month = dict(sorted(pars_by_month.items()))
    p0_by_month = dict(sorted(p0_by_month.items()))
    corCTS_by_month = dict(sorted(corCTS_by_month.items()))
    corBTS_by_month = dict(sorted(corBTS_by_month.items()))

    sim_data = data[["Time", "Value"]].copy()
    sim_data["month"] = sim_data["Time"].dt.month
    sim_data["year"] = sim_data["Time"].dt.year

    start_year = sim_data["year"].min()
    years = np.arange(start_year, start_year + n_years)
    months = np.arange(1, 13)
    days_in_month = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])

    nn = np.tile(days_in_month, (len(years), 1))

    BTs = np.random.normal(0, 1, p)
    CTs = np.random.normal(0, 1, p)

    dim0 = []
    for row in nn:
        dim0.extend(row)

    dim = np.concatenate([[p], dim0])
    dim2 = np.cumsum(dim)
    dim1 = dim2 - p

    ind_to_add = np.arange(0, len(years) * len(months), len(months)).tolist()

    for y_idx, year in enumerate(years):
        for month in range(1, len(months) + 1):
            n = nn[y_idx, month - 1]

            # Binary process
            corGBTS = actf_binary(corBTS_by_month[month], p0_by_month[month])

            P = np.zeros((p, p))
            for j in range(p):
                for i in range(p):
                    ind = abs(i - j)
                    P[j, i] = corGBTS[ind]
            Pinv = np.linalg.inv(P)
            corGBTS2 = corGBTS[1:(p + 2)]
            alpha_BTs = Pinv.dot(corGBTS2)

            alpha_flipped_BTs = np.flipud(alpha_BTs)
            sigma_eps_BTs = math.sqrt(max(1 - np.sum(alpha_BTs * corGBTS2), 0))

            for k in range(n):
                eps_BTs = np.random.normal(0.0, sigma_eps_BTs, 1)
                start = int(dim1[(month - 1) + ind_to_add[y_idx]] + k)
                stop = int(dim2[(month - 1) + ind_to_add[y_idx]] + k)
                to_be_add_BTs = np.sum(BTs[start:stop] * alpha_flipped_BTs) + eps_BTs
                BTs = np.concatenate((BTs, to_be_add_BTs))

            # Continuous process
            corGCTS = np.array([2 * math.sin((math.pi / 6) * value) for value in corCTS_by_month[month]])

            P_cts = np.zeros((p, p))
            for j in range(p):
                for i in range(p):
                    ind = abs(i - j)
                    P_cts[j, i] = corGCTS[ind]

            P_ctsinv = np.linalg.inv(P_cts)
            corGCTS2 = corGCTS[1:(p + 1)]
            alpha_CTs = P_ctsinv.dot(corGCTS2)

            alpha_flipped_CTs = np.flipud(alpha_CTs)
            sigma_eps_CTs = math.sqrt(max(1 - np.sum(alpha_CTs * corGCTS2), 0))

            for k in range(n):
                eps_CTs = np.random.normal(0.0, sigma_eps_CTs, 1)
                start = int(dim1[(month - 1) + ind_to_add[y_idx]] + k)
                stop = int(dim2[(month - 1) + ind_to_add[y_idx]] + k)
                to_be_add_CTs = np.sum(CTs[start:stop] * alpha_flipped_CTs) + eps_CTs
                CTs = np.concatenate((CTs, to_be_add_CTs))

    GBTS = pd.DataFrame(BTs[p:], columns=["Gauss_Ts"])
    GCTS = pd.DataFrame(CTs[p:], columns=["Gauss_Ts"])

    start_date = pd.to_datetime(sim_data.iloc[0, 0])
    start_date = pd.Timestamp(year=start_date.year, month=1, day=1)

    n_days_target = 365 * n_years
    buffer_days = int(n_years / 4) + 5
    date_range_full = pd.date_range(start=start_date, periods=n_days_target + buffer_days, freq="D")
    date_range_clean = date_range_full[~((date_range_full.month == 2) & (date_range_full.day == 29))]
    date = date_range_clean[:n_days_target]

    m = pd.DatetimeIndex(date).month
    y = pd.DatetimeIndex(date).year

    GBTS["Time"] = date
    GBTS["month"] = m
    GBTS["year"] = y

    GCTS["Time"] = date
    GCTS["month"] = m
    GCTS["year"] = y

    sim_bts = []
    sim_cts = []

    for year in years:
        for month in range(1, len(months) + 1):
            temp_GBTs = GBTS[(GBTS["month"] == month) & (GBTS["year"] == year)]
            zm_GBTs = temp_GBTs.iloc[:, [0]]

            temp_GCTs = GCTS[(GCTS["month"] == month) & (GCTS["year"] == year)]
            zm_GCTs = temp_GCTs.iloc[:, [0]]

            z_p0_m = stats.norm.ppf(p0_by_month[month], loc=0, scale=1)

            BTSsim = np.where(zm_GBTs < z_p0_m, 0, 1)
            CTSsim = stats.gamma.ppf(
                stats.norm.cdf(zm_GCTs, loc=0, scale=1),
                a=pars_by_month[month][0],
                loc=pars_by_month[month][1],
                scale=pars_by_month[month][2],
            )

            sim_bts.append(pd.DataFrame(BTSsim, columns=["Value"]))
            sim_cts.append(pd.DataFrame(CTSsim, columns=["Value"]))

    simulated_bts = pd.concat(sim_bts, axis=0).reset_index(drop=True)
    simulated_cts = pd.concat(sim_cts, axis=0).reset_index(drop=True)

    sim_ts = simulated_bts.iloc[:, 0] * simulated_cts.iloc[:, 0]

    SimTs = pd.DataFrame({
        #"Time": date,
        "Value": sim_ts
    })

    # ------------------------------------------------------------------
    # Final output
    # ------------------------------------------------------------------
    return {
        "simulated_ts": SimTs
    }

In [58]:
df = pd.read_csv("daily_data.csv")   # due colonne: Time, Value
res = CoSMoS2s(data=df)